# Modeling and Evaluation
Goal: predict Productivity_Score using four regression models and compare them.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

## 1. Load the data

In [2]:
df = pd.read_csv("../Data/Cleaned/cleaned_data.csv")
print(df.shape)
df.head()

(288, 19)


,User_ID,Age,Gender,Country,User_Type,Education_Level,Profession,Monthly_Income,AI_Tool,AI_Purpose,AI_Usage_Hours_Per_Day,Tasks_Performed,Monthly_AI_Cost,Productivity_Score,Accuracy_Rating,Satisfaction_Score,Time_Saved_Hours_Per_Week,Work_or_Study_Hours_Per_Day,Would_Recommend
0,U0269,30.0,Male,United Kingdom,Professional,Master's,Business Analyst,5421.0,ChatGPT,Office Work,2.2,Data analysis using AI-powered analytics tools...,0.0,5.3,4.0,7.0,3.9,7.3,Yes
1,U0217,30.0,Male,United States,Professional,Bachelor's,Customer Support Specialist,8903.0,ChatGPT,Customer Support,5.7,Resolving customer inquiries using AI chatbot ...,0.0,9.9,3.0,9.0,11.4,7.2,Yes
2,U0152,17.0,Male,Canada,Student,Unknown,Computer Science Student,402.0,ChatGPT,Coding,2.2,Unknown,20.0,3.2,2.0,4.0,5.2,6.7,No
3,U0264,35.0,Female,Brazil,Professional,Bachelor's,Desenvolvedora de Software / Software Developer,1367.0,Meta AI,Coding,1.7,"Usar GitHub Copilot para autocompletar código,...",20.0,5.6,4.0,5.0,6.4,9.6,Maybe
4,U0263,19.0,Female,Canada,Student,Undergraduate,Biology Student,438.0,Perplexity AI,Unknown,4.1,Using AI to generate study summaries for exam ...,0.0,4.7,4.0,7.0,9.2,6.2,Yes


## 2. Check the data

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 288 entries, 0 to 287
Data columns (total 19 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   User_ID                      288 non-null    str    
 1   Age                          288 non-null    float64
 2   Gender                       288 non-null    str    
 3   Country                      288 non-null    str    
 4   User_Type                    288 non-null    str    
 5   Education_Level              288 non-null    str    
 6   Profession                   288 non-null    str    
 7   Monthly_Income               288 non-null    float64
 8   AI_Tool                      288 non-null    str    
 9   AI_Purpose                   288 non-null    str    
 10  AI_Usage_Hours_Per_Day       288 non-null    float64
 11  Tasks_Performed              288 non-null    str    
 12  Monthly_AI_Cost              288 non-null    float64
 13  Productivity_Score           27

In [4]:
df.isnull().sum()

User_ID                         0
Age                             0
Gender                          0
Country                         0
User_Type                       0
Education_Level                 0
Profession                      0
Monthly_Income                  0
AI_Tool                         0
AI_Purpose                      0
AI_Usage_Hours_Per_Day          0
Tasks_Performed                 0
Monthly_AI_Cost                 0
Productivity_Score             14
Accuracy_Rating                 0
Satisfaction_Score              0
Time_Saved_Hours_Per_Week       0
Work_or_Study_Hours_Per_Day     0
Would_Recommend                 0
dtype: int64

**Conclusion:** Productivity_Score has 14 missing values (about 5%). Accuracy_Rating has none. Rows without a target value cannot be used for training or testing, so they are removed.

In [5]:
df =df.dropna(subset=["Productivity_Score"]).reset_index(drop=True)
print(df.shape)

(274, 19)


## 3. Target and features

In [6]:
target= "Productivity_Score"
print(df[target].describe())
y=df[target]
X= df.drop(columns=[target,"User_ID"])
print(X.shape, y.shape)

count    274.000000
mean       5.235036
std        1.821789
min        1.600000
25%        3.900000
50%        5.000000
75%        6.500000
max       10.000000
Name: Productivity_Score, dtype: float64
(274, 17) (274,)


**Conclusion:**Productivity_Score is targeted as it is a continuous, meaningful variable .so it is used as the target. Its 14 missing rows were removed in step 2. User_ID is dropped to prevent the model from memorizing rows instead of learning general patterns.

## 4. Convert text columns to numbers

Models only understand numbers, so the text columns must be converted first.

In [7]:
text_cols = X.select_dtypes(exclude=["number"])
print(text_cols.nunique().sort_values (ascending=False))

Tasks_Performed    259
Profession         123
Country             13
AI_Tool              9
AI_Purpose           9
Education_Level      7
Gender               5
Would_Recommend      4
User_Type            2
dtype: int64


**Conclusion:** Tasks_Performed (259 unique values) and Profession (123 unique values) would create hundereds of columns for only 274 rows, so both are dropped. The other text columns have few values and are kept.

In [8]:
X = X.drop(columns=["Tasks_Performed","Profession"])
print(X.shape)

(274, 15)


### One-hot encoding
Each text column becomes several 0/1 columns.  'drop_firt=True' removes one category per column to avoid repeating the same information.

In [9]:
X = pd.get_dummies(X, drop_first=True)
print(X.shape)
X.head()

(274, 50)


,Age,Monthly_Income,AI_Usage_Hours_Per_Day,Monthly_AI_Cost,Accuracy_Rating,Satisfaction_Score,Time_Saved_Hours_Per_Week,Work_or_Study_Hours_Per_Day,Gender_Male,Gender_Non-Binary,...,AI_Purpose_Customer Support,AI_Purpose_Data Analysis,AI_Purpose_Office Work,AI_Purpose_Research,AI_Purpose_Studying,AI_Purpose_Unknown,AI_Purpose_Writing,Would_Recommend_No,Would_Recommend_Unknown,Would_Recommend_Yes
0,30.0,5421.0,2.2,0.0,4.0,7.0,3.9,7.3,True,False,...,False,False,True,False,False,False,False,False,False,True
1,30.0,8903.0,5.7,0.0,3.0,9.0,11.4,7.2,True,False,...,True,False,False,False,False,False,False,False,False,True
2,17.0,402.0,2.2,20.0,2.0,4.0,5.2,6.7,True,False,...,False,False,False,False,False,False,False,True,False,False
3,35.0,1367.0,1.7,20.0,4.0,5.0,6.4,9.6,False,False,...,False,False,False,False,False,False,False,False,False,False
4,19.0,438.0,4.1,0.0,4.0,7.0,9.2,6.2,False,False,...,False,False,False,False,False,True,False,False,False,True


**Conclusion:** After encoding, X has 50 columns.

## 5. Train/test split
80% of the data is used for tranining and 20% for testing. 'random_state=42' makes the split repeatable.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(219, 50) (55, 50)


**Conclusion:** The training set has 219 rows and the test set has 55 rows, both with 50 columns.

## 6. Check for missing values
After removing the rows with a missing target, no missing values should remain in the features or the target.

In [14]:
print(X_train.isnull().sum().sum(), X_test.isnull().sum().sum(), y_train.isnull().sum(), y_test.isnull().sum())

0 0 0 0


**Conclusion:** All four counts are 0, so the data is ready for scaling and modeling.